# QUANTIZATION-AWARE TRAINING (QAT)

    This notebook demonstrates QUANTIZATION-AWARE TRAINING (QAT), an advanced
    quantization technique that simulates quantization DURING training.
    
    KEY DIFFERENCE FROM PTQ (Post-Training Quantization):
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    POST-TRAINING QUANTIZATION (PTQ):
        Train (FP32) → Calibrate → Quantize (INT8)
        ✓ Fast and easy
        ✗ May lose 1-3% accuracy
    
    QUANTIZATION-AWARE TRAINING (QAT):
        Train with simulated quantization → Quantize (INT8)
        ✓ Better accuracy (<1% drop)
        ✓ Model learns to be robust to quantization
        ✗ Requires retraining (more expensive)
    
    THE BIG IDEA: During training, simulate the quantization noise so the model
    learns weights that work well even after being quantized!
    
    Visual comparison:
        
    PTQ: [Train in FP32] -> [Surprise Now you're INT8] -> Accuracy drop
                                ↑
                        Model wasn't prepared!
    
    QAT: [Train with fake quantization] -> [Convert to INT8] -> ✓ Minimal drop
                    ↓                              ↑
            Model adapts during training   Already prepared!

In [1]:
import torch
import torchvision.datasets as datasets 
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import os

# Make torch deterministic for reproducible results
_ = torch.manual_seed(0)

# Normalize MNIST images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST training set
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load MNIST test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Use CPU (quantization works on CPU)
device = "cpu"

## DEFINE MODEL WITH QUANTIZATION STUBS

In [2]:
class VerySimpleNet(nn.Module):
    """
    Neural network WITH quantization stubs built-in from the start.
    
    Important difference from PTQ:
    - In PTQ: We add stubs AFTER training
    - In QAT: We add stubs BEFORE training
    
    Because QAT needs to simulate quantization during training!
    
    Architecture with quantization boundaries:
    
    FP32 Input
        ↓
    [QuantStub] <- Simulates quantization: FP32 -> fake INT8 -> FP32
        ↓ (during training: adds quantization noise)
    Linear1 + ReLU
        ↓
    Linear2 + ReLU
        ↓
    Linear3
        ↓
    [DeQuantStub] <- Marks end of quantized region
        ↓
    FP32 Output
    """
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(VerySimpleNet, self).__init__()
        
        # Quantization stub - will simulate INT8 quantization during training
        self.quant = torch.quantization.QuantStub()
        
        self.linear1 = nn.Linear(28*28, hidden_size_1) 
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2) 
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()
        
        # Dequantization stub - marks end of quantized region
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, img):
        x = img.view(-1, 28*28)
        # During QAT training: fake quantize input (adds noise)
        x = self.quant(x)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        # During QAT training: no-op (but marks boundary)
        x = self.dequant(x)
        return x

net = VerySimpleNet().to(device)

## PREPARE MODEL FOR QAT (INSERT FAKE QUANTIZATION)

In [3]:
# Set quantization configuration
net.qconfig = torch.ao.quantization.default_qconfig

# Set to training mode
net.train()

# CRITICAL: prepare_qat() instead of prepare()
# This inserts "fake quantization" modules
net_quantized = torch.ao.quantization.prepare_qat(net)
print(net_quantized)

VerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=inf, max_val=-inf)
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=inf, max_val=-inf)
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=inf, max_val=-inf)
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)


    prepare_qat() inserted TWO types of observers per layer:
    
    1. weight_fake_quant: MinMaxObserver
       - Observes weight values during training
       - Simulates quantizing weights to INT8
       
    2. activation_post_process: MinMaxObserver  
       - Observes activation values during training
       - Simulates quantizing activations to INT8
    
    Example for linear1:
        Linear(
            weight_fake_quant: MinMaxObserver(min_val=inf, max_val=-inf)
            activation_post_process: MinMaxObserver(min_val=inf, max_val=-inf)
        )
    
    FAKE QUANTIZATION:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    During training, fake quantization does this for each weight:
    
        Original weight (FP32): 0.0234
                ↓ (quantize to INT8)
        Quantized: round(0.0234 / 0.0043) = 5
                ↓ (dequantize back to FP32)
        Fake quantized weight: 5 × 0.0043 = 0.0215
                ↓
        Use this in forward pass!
    
    Visual representation:
        
        Real weight: 0.0234
                       ↓ (simulate quantization noise)
        Noisy weight: 0.0215  ← This is what INT8 will give us!
                       ↓
        Backprop updates the REAL weight to minimize loss with this noise
    
    The model learns to work well DESPITE quantization errors!
    
    KEY INSIGHT: By training with quantization noise, the model adapts:
    - Weights move to values that quantize cleanly
    - Network becomes robust to precision loss
    - Final INT8 model performs much better than PTQ!

## TRAINING FUNCTIONS

In [4]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    """
    Standard training loop, but with fake quantization happening inside!
    
    During each forward pass:
    1. Weights are fake-quantized (FP32 → INT8 → FP32)
    2. Activations are fake-quantized
    3. Loss is computed on fake-quantized outputs
    4. Gradients flow back through fake quantization
    5. Real FP32 weights are updated
    
    The model learns to minimize loss WHILE being quantized!
    """
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()  # Enable fake quantization

        loss_sum = 0
        num_iterations = 0

        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
            
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            
            # Forward pass with FAKE QUANTIZATION
            optimizer.zero_grad()
            output = net(x.view(-1, 28*28))  # Weights/activations are fake-quantized here!
            loss = cross_el(output, y)
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            
            # Backward pass - gradients account for quantization noise
            loss.backward()
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

            
def print_size_of_model(model):
    """Print the size of the model in KB."""
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

    Example weight update with QAT:
    
    Iteration 1:
        Weight: 0.0234 (FP32)
            ↓ fake quantize
        Fake: 0.0215 (simulated INT8)
            ↓ forward pass
        Loss: 2.3
            ↓ backward pass  
        Gradient considers quantization noise!
            ↓ update
        New weight: 0.0245
        
    Iteration 2:
        Weight: 0.0245
            ↓ fake quantize
        Fake: 0.0258 (different quantization point)
            ↓ forward pass
        Loss: 2.1 ← Getting better!
        
    Over many iterations:
        Weights converge to values that:
        ✓ Minimize loss
        ✓ Quantize well (minimal error from quantization)
        ✓ Make the network robust to INT8 precision
    
    Compare to PTQ:
        PTQ trains without quantization noise
        → Weights might land on "unlucky" values that quantize poorly
        → Accuracy drop when converted to INT8
        
        QAT trains WITH quantization noise  
        → Weights land on "lucky" values that quantize well
        → Minimal accuracy drop when converted to INT8

In [5]:
# Train the model with QAT
print("Training with Quantization-Aware Training...")
train(train_loader, net_quantized, epochs=1)

Training with Quantization-Aware Training...


Epoch 1: 100%|████████████████████████████████████████████████████████| 6000/6000 [00:31<00:00, 189.68it/s, loss=0.224]


## TESTING FUNCTION

In [6]:
def test(model: nn.Module, total_iterations: int = None):
    """Evaluate model accuracy on test set."""
    correct = 0
    total = 0
    iterations = 0

    model.eval()  # Disable fake quantization (use actual quantized values)

    with torch.no_grad():
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = model(x.view(-1, 784))
            
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct += 1
                total += 1
                
            iterations += 1
            if total_iterations is not None and iterations >= total_iterations:
                break
                
    print(f'Accuracy: {round(correct/total, 3)}')

## CHECK COLLECTED STATISTICS DURING TRAINING

In [7]:
print(f'Check statistics of the various layers')
print(net_quantized)

Check statistics of the various layers
VerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=-0.4242129623889923, max_val=2.821486711502075)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=-0.5647792816162109, max_val=0.33721768856048584)
    (activation_post_process): MinMaxObserver(min_val=-42.90361404418945, max_val=35.304378509521484)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=-0.43838435411453247, max_val=0.33769407868385315)
    (activation_post_process): MinMaxObserver(min_val=-41.151580810546875, max_val=22.324588775634766)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (weight_fake_quant): MinMaxObserver(min_val=-0.427390456199646, max_val=0.2056414932012558)
    (activation_post_process): MinMaxObserver(min_val=-31.895679473876953, max_val=22.713518142700195)


    Notice we have TWO sets of statistics per layer:
    
    1. weight_fake_quant: MinMaxObserver(min_val=-0.549, max_val=0.307)
       These are the WEIGHT ranges observed during training
       Used to quantize the weights themselves
       
    2. activation_post_process: MinMaxObserver(min_val=-42.47, max_val=40.64)
       These are the ACTIVATION ranges observed during training
       Used to quantize the layer outputs
    
    Example for linear1:
        weight_fake_quant: [-0.549, 0.307]
        → Weight scale = (0.307 - (-0.549)) / 255 = 0.00335
        → Weights will be quantized using this scale
        
        activation_post_process: [-42.47, 40.64]  
        → Activation scale = (40.64 - (-42.47)) / 255 = 0.326
        → Activations will be quantized using this scale
    
    KEY DIFFERENCE FROM PTQ:
    - PTQ: Only collects activation statistics (weights are fixed)
    - QAT: Collects BOTH weight and activation statistics (everything adapts)
    
    The weights were TRAINED to work well within these ranges!

## CONVERT TO REAL INT8 MODEL

In [8]:
# Set to eval mode before converting
net_quantized.eval()

# Convert fake quantization to real INT8
net_quantized = torch.ao.quantization.convert(net_quantized)

    torch.ao.quantization.convert() performs the final conversion:
    
    1. Removes fake quantization modules
    2. Converts FP32 weights → INT8 using observed ranges
    3. Converts FP32 layers → QuantizedLinear layers
    4. Calculates final scale/zero_point for each layer
    
    Before convert():
        Linear layer with fake quantization:
        - FP32 weights: [-0.549, ..., 0.307]
        - Fake quantize applied during forward pass
        
    After convert():
        QuantizedLinear:
        - INT8 weights: [-127, ..., 71]
        - scale: 0.00335
        - zero_point: 0
        - Real INT8 operations!
    
    The difference from PTQ:
        PTQ: Weights not trained for quantization → larger errors
        QAT: Weights TRAINED for quantization → smaller errors ✓

In [9]:
print(f'Check statistics of the various layers')
print(net_quantized)

Check statistics of the various layers
VerySimpleNet(
  (quant): Quantize(scale=tensor([0.0256]), zero_point=tensor([17]), dtype=torch.quint8)
  (linear1): QuantizedLinear(in_features=784, out_features=100, scale=0.6158109903335571, zero_point=70, qscheme=torch.per_tensor_affine)
  (linear2): QuantizedLinear(in_features=100, out_features=100, scale=0.49981236457824707, zero_point=82, qscheme=torch.per_tensor_affine)
  (linear3): QuantizedLinear(in_features=100, out_features=10, scale=0.4299936890602112, zero_point=74, qscheme=torch.per_tensor_affine)
  (relu): ReLU()
  (dequant): DeQuantize()
)


## EXAMINE QUANTIZED WEIGHTS

In [10]:
print('Weights after quantization')
print(torch.int_repr(net_quantized.linear1.weight()))

Weights after quantization
tensor([[ 5,  9, -2,  ..., 10,  5,  5],
        [-5, -4, -3,  ..., -5, -2, -7],
        [-1,  7, -4,  ..., -1,  4,  5],
        ...,
        [ 9, 10,  2,  ...,  0,  5, -4],
        [-3, -2,  6,  ...,  1,  1,  1],
        [ 5,  4,  1,  ...,  9, -2,  2]], dtype=torch.int8)


    Key observation: These weights were TRAINED to work well in INT8!
    
    During QAT training:
        The model learned that weight=0.0234 would become INT8=5
        It adjusted other weights knowing this quantization would happen
        The final network is optimized for INT8 representation
    
    Compare to PTQ:
        PTQ: Weights trained in FP32, converted blindly to INT8
        QAT: Weights trained KNOWING they'd become INT8 ✓
        
    This is why QAT typically gives better accuracy!

## TEST FINAL QUANTIZED MODEL

In [11]:
print('Testing the model after quantization')
test(net_quantized)

Testing the model after quantization


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:02<00:00, 453.94it/s]

Accuracy: 0.951


    In this simple example, both PTQ and QAT work well because:
    - The network is small and simple
    - MNIST is an easy dataset
    - There's plenty of capacity

    QAT becomes essential for:
    ✓ Complex models (ResNet, BERT, etc.)
    ✓ Aggressive quantization (4-bit, 2-bit)
    ✓ Models close to capacity (where every bit matters)
    ✓ Edge deployment with strict accuracy requirements
        
    For production deployment where 1-2% accuracy matters, QAT wins

    QUANTIZATION METHOD COMPARISON:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    POST-TRAINING QUANTIZATION (PTQ):
    ✓ No retraining needed
    ✓ Fast to implement (minutes)
    ✓ Good for simple models
    ✓ 1-3% accuracy drop typical
    ✗ Not optimal for complex models
    ✗ Weights not adapted to quantization
    
    QUANTIZATION-AWARE TRAINING (QAT):
    ✓ Better accuracy (<1% drop)
    ✓ Weights trained to quantize well
    ✓ Essential for aggressive quantization (4-bit, 2-bit)
    ✓ Best for production deployment
    ✗ Requires retraining (hours/days)
    ✗ More complex to implement
    
    THE COMPLETE PICTURE:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    1. BASELINE: Train FP32 model
       Size: 360 KB | Speed: 1x | Accuracy: 96%
    
    2. PTQ: Quantize after training
       Size: 95 KB | Speed: 2-4x | Accuracy: 96% (0% drop)
       Time: Minutes
    
    3. QAT: Train with quantization
       Size: 95 KB | Speed: 2-4x | Accuracy: 95% (-1% drop)*
       Time: Full training + overhead
       
    *Note: For complex models, QAT usually BEATS PTQ in accuracy!
    
    4. OTHER TECHNIQUES:
       - Mixed precision: Critical layers in FP16/FP32, rest in INT8
       - Dynamic quantization: Weights INT8, activations FP32
       - Progressive quantization: Gradually increase quantization during training